# Customer Churn Prediction
This notebook covers the end-to-end pipeline: 
- Data Loading & EDA
- Preprocessing & Feature Engineering
- Modeling & Evaluation
- Saving the best model for deployment

## 1️⃣ Load Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import pickle

sns.set(style='whitegrid')

## 2️⃣ Load Dataset

In [ ]:
# Load raw dataset
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## 3️⃣ Explore Data

In [ ]:
# Info & missing values
df.info()
df.isnull().sum()

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Visualize target variable
sns.countplot(x='Churn', data=df)
plt.title('Churn Distribution')
plt.show()

In [ ]:
# Heatmap for correlation (numeric only)
numeric_cols = df.select_dtypes(include=['float64','int64']).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.show()

## 4️⃣ Data Cleaning & Preprocessing

In [ ]:
import os

# Drop ID column (not useful for prediction)
df.drop('customerID', axis=1, inplace=True)

# Convert TotalCharges to numeric (it has spaces)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Encode target variable
df['Churn_Yes'] = df['Churn'].apply(lambda x: 1 if x=='Yes' else 0)
df.drop('Churn', axis=1, inplace=True)

# Encode categorical variables (exclude high-cardinality identifiers)
cat_cols = df.select_dtypes(include=['object']).columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Scale numeric columns
scaler = StandardScaler()
num_cols = ['tenure','MonthlyCharges','TotalCharges']
df[num_cols] = scaler.fit_transform(df[num_cols])

# Create processed directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save cleaned dataset
df.to_csv('../data/processed/cleaned_telco_churn.csv', index=False)

print("Processed dataset saved successfully.")

## 5️⃣ Split Data

In [ ]:
X = df.drop('Churn_Yes', axis=1)
y = df['Churn_Yes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 6️⃣ Train Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=500),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    results[name] = {'Accuracy': acc, 'F1': f1, 'ROC_AUC': roc}
    print(f"{name}: Accuracy={acc:.3f}, F1={f1:.3f}, ROC_AUC={roc:.3f}")

## 7️⃣ Feature Importance (Random Forest Example)

In [ ]:
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
feat_names = X.columns

feat_importance = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
feat_importance.sort_values(by='Importance', ascending=False, inplace=True)

plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=feat_importance.head(20))
plt.title('Top 20 Feature Importances')
plt.show()

## 8️⃣ Save Best Model

In [ ]:
# Save Random Forest as the best model (or choose your best)
pickle.dump(rf_model, open('../model/churn_model.pkl', 'wb'))